In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

In [ ]:
import os
import sys
import json
import logging

import torch
import transformers

from rich.console import Console
from rich.theme import Theme

from src.utils import logging_utils

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,7"

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, highlight=False)

def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)

logging.basicConfig(
    level=logging.INFO,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)
logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(f"{transformers.__version__=}")

In [ ]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

In [ ]:
from src.goals.head_loc_attn import HeadMask
from src.utils import env_utils

MASK_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    f"localization/{MODEL_NAME.split('/')[-1]}/head_mask.pt",
)

ckpt = torch.load(MASK_PATH, map_location="cpu")
print(f"Loaded checkpoint from {MASK_PATH}")
# print(f"  Trained for {ckpt['config']['n_epochs']} epochs  |  N={ckpt['config']['n_instances']} instances")
# print(f"  λ_sparse={ckpt['config']['lambda_sparse']}  lr={ckpt['config']['lr']}")

# Reconstruct mask object
LAYERS = ckpt["layers"]
n_layers_tested = len(LAYERS)
n_heads = ckpt["mask_values"].shape[1]
goalscope_mask = HeadMask(n_layers_tested, n_heads)
with torch.no_grad():
    goalscope_mask.values.copy_(ckpt["mask_values"])
    if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        goalscope_mask.values[0:25] = 0  # zero out early layers
        goalscope_mask.values[56:] = 0  # zero out late layers
    if mt.name.split("/")[-1] == "Llama-3.3-70B-Instruct":
        goalscope_mask.values[0:18] = 0  # zero out early layers
        goalscope_mask.values[45:] = 0  # zero out late layers

threshold = 0.9
goalscope_heads = goalscope_mask.selected_heads(LAYERS, threshold=threshold)
print(
    f"\n  Layers: {LAYERS[0]}–{LAYERS[-1]}  ({n_layers_tested} tested, {n_heads} heads each)"
)
print(f"  Active heads (mask > {threshold}): {len(goalscope_heads)}")
print(f"  Selected heads: {goalscope_heads}")

_n_lt = len(LAYERS)
_n_h = mt.config.num_attention_heads
goalscope_binary_mask = HeadMask(_n_lt, _n_h)
with torch.no_grad():
    goalscope_binary_mask.values.zero_()
    for _layer_idx, _head_idx in goalscope_heads:
        goalscope_binary_mask.values[LAYERS.index(_layer_idx), _head_idx] = 1.0

print(
    f"Binary mask: {goalscope_binary_mask.n_active()} active heads / {_n_lt * _n_h} total heads "
    f"({100 * goalscope_binary_mask.n_active() / (_n_lt * _n_h):.2f}%)"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

filter_head_path = optimized_path = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "filter",
    mt.name.split("/")[-1],
    "distinct_options/select_one/legacy/epoch_5.npz"
)

filter_results_raw = np.load(optimized_path, allow_pickle=True)
# plt.plot(optimization_results["losses"])
# plt.show()

filter_mask = HeadMask(n_layers_tested, n_heads)
with torch.no_grad():
    filter_mask.values.copy_(torch.tensor(filter_results_raw["optimal_mask"]).to(torch.float32))
    # if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
    #     filter_mask.values[0:25] = 0  # zero out early layers
    #     filter_mask.values[56:] = 0  # zero out late layers
    # if mt.name.split("/")[-1] == "Llama-3.3-70B-Instruct":
    #     filter_mask.values[0:18] = 0  # zero out early layers
    #     filter_mask.values[45:] = 0  # zero out late layers


threshold = 0.9
filter_heads = filter_mask.selected_heads(LAYERS, threshold=threshold)
print(
    f"\n  Layers: {LAYERS[0]}–{LAYERS[-1]}  ({n_layers_tested} tested, {n_heads} heads each)"
)
print(f"  Active heads (mask > {threshold}): {len(filter_heads)}")
print(f"  Selected heads: {filter_heads}")

_n_lt = len(LAYERS)
_n_h = mt.config.num_attention_heads
filter_binary_mask = HeadMask(_n_lt, _n_h)
with torch.no_grad():
    filter_binary_mask.values.zero_()
    for _layer_idx, _head_idx in filter_heads:
        filter_binary_mask.values[LAYERS.index(_layer_idx), _head_idx] = 1.0

print(
    f"Binary mask: {filter_binary_mask.n_active()} active heads / {_n_lt * _n_h} total heads "
    f"({100 * filter_binary_mask.n_active() / (_n_lt * _n_h):.2f}%)"
)

In [ ]:
### Visualize overlap — analog of filter/001_make_figures.ipynb

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap


def get_ticks(ticks, skip=5):
    return [str(i) if i % skip == 0 else "" for i in ticks]


n_layer = mt.n_layer
n_heads = mt.model.config.num_attention_heads

gs_set = set(goalscope_heads)
fl_set = set(filter_heads)
overlap = list(gs_set & fl_set)
print(f"goalscope: {len(gs_set)} | filter: {len(fl_set)} | overlap: {len(overlap)}")

goalscope_mask_arr = torch.zeros((n_heads, n_layer), dtype=torch.float32)
for layer_idx, head_idx in goalscope_heads:
    goalscope_mask_arr[head_idx, layer_idx] = 1.0

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "Times New Roman"

SMALL_SIZE = 16
MEDIUM_SIZE = 20
BIGGER_SIZE = 24

plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=BIGGER_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

fig, ax = plt.subplots(figsize=(16, 9))

cmap = LinearSegmentedColormap.from_list("white_orange", ["white", "orange"], N=100)
ax.imshow(goalscope_mask_arr.numpy(), cmap=cmap, aspect="auto", vmin=0, vmax=1, alpha=0.7)

for layer_idx, head_idx in filter_heads:
    ax.add_patch(
        patches.Rectangle(
            (layer_idx - 0.5, head_idx - 0.5),
            1, 1,
            linewidth=1.0,
            edgecolor="black",
            facecolor="none",
        )
    )

for layer_idx, head_idx in overlap:
    ax.add_patch(
        patches.Rectangle(
            (layer_idx - 0.5, head_idx - 0.5),
            1, 1,
            linewidth=2.0,
            edgecolor="red",
            facecolor="none",
        )
    )

plt.xlabel("Layer Index")
plt.ylabel("Head Index")
plt.xticks(ticks=range(n_layer), labels=get_ticks(range(n_layer), skip=4), rotation=45)
plt.yticks(ticks=range(n_heads), labels=get_ticks(range(n_heads), skip=4))

legend_elements = [
    mpatches.Patch(facecolor="orange", alpha=0.7, label=f"Goalscope Heads ({len(gs_set)})"),
    mpatches.Patch(facecolor="white", edgecolor="black", linewidth=1, label=f"Filter Heads ({len(fl_set)})"),
    mpatches.Patch(facecolor="white", edgecolor="red", linewidth=2.5, label=f"Overlap ({len(overlap)})"),
]
ax.legend(
    handles=legend_elements,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.10),
    ncol=3,
    fontsize=MEDIUM_SIZE,
    frameon=False,
)

plt.tight_layout()
plt.show()

In [ ]:
from src.goals.data import IncompleteCodeSample
from src.goals.functional import supports_chat_template, apply_chat_template
from src.goals.goalscope import greedy_continue
from rich.markup import escape
from src.tokens import prepare_input

def get_prompt(sample, mt=mt):
    if supports_chat_template(mt):
        prompt = apply_chat_template(
            mt,
            user_query=sample.full_instruction.strip(),
            incomplete_code=sample.code,
        )
    else:
        prompt = sample.code
    return prompt

test_samples = [
    IncompleteCodeSample(
        instruction="""Write a python code that does the following:
1. reads a .json file from a give local file path. the json file contains a list of objects.
2. indexes the objects in the json file by a specific key for fast retrieval.
3. given a target value, retrieves the corresponding object from the indexed data and returns it. time complexity of retrival should be O(log N) or better. 
    * make sure to handle error cases, such as when the target value is not found in the indexed data.
4. take user inputs, from the terminal, for the target value. assume multiple queries will be made.

Write your code in a modular way, with clear function definitions.
Do not add any comments or docstrings. Only provide the code.""",
        code="",
        unit_tests=[],
        function_name="",
        prefills=[],
        augmentation=None,
    ),
]

test_sample = test_samples[-1]
test_prompt = get_prompt(test_sample, mt)

test_gen = greedy_continue(
    mt,
    **prepare_input(prompts=test_prompt, tokenizer=mt),
    max_new_tokens=512,
)
rich_print(f"[white]{escape(test_prompt)}[/white][sea_green3]{escape(test_gen)}[/sea_green3]")

In [ ]:
full_prompt = test_prompt + test_gen

In [22]:
### Interactive attention visualization
# Hover over a token to see its attention distribution (aggregated over
# selected_heads) over the previous token positions. Click to pin/unpin.
# value_weighted=True scales each (q, k) by ||v_k|| for actual residual contribution.
#
# Consumes `full_prompt` from the previous cell (prefix + generated_code).

from src.attention import get_attention_matrices, show_interactive_attention
from src.tokens import prepare_input

# Batch the same prompt twice as a smoke-test for batched get_attention_matrices.
tokenized = prepare_input([full_prompt, full_prompt], tokenizer=mt)
attn_infos = get_attention_matrices(tokenized, mt, value_weighted=True)
print(f"Batched: returned {len(attn_infos)} AttentionInformation objects")
attn_info = attn_infos[0]
print(f"  attention_matrices.shape = {attn_info.attention_matrices.shape}")
print(f"  n_tokens = {len(attn_info.tokenized_prompt)}")

out_dir = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "interactive_attn", mt.name.split("/")[-1]
)
os.makedirs(out_dir, exist_ok=True)
# out_path = os.path.join(out_dir, "goalscope_clock_multistep.html")

###########################################
selected_heads = filter_heads
###########################################

show_interactive_attention(
    attn_info=attn_info,
    selected_heads=selected_heads,
    start_from=2,
    agg="mean",
    normalize="row_max",
    cmap="Blues",
    title=f"Selected heads ({len(selected_heads)}) · value-weighted attention",
    # save_path=out_path,
    height=None,
)
# print(f"Saved → {out_path}")

Batched: returned 2 AttentionInformation objects
  attention_matrices.shape = (64, 40, 479, 479)
  n_tokens = 479


'<!DOCTYPE html>\n<html><head><meta charset="utf-8"><title>Selected heads (54) · value-weighted attention</title>\n<style>\n #iaviz-389d0815 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n            background: #fafafa; color: #111; padding: 12px; border-radius: 6px; }\n #iaviz-389d0815 h2 { margin: 4px 0 6px 0; font-size: 16px; font-weight: 600; color: #111; }\n #iaviz-389d0815 .meta { color: #555; font-size: 12px; margin-bottom: 10px; }\n #iaviz-389d0815 .head-list { color: #777; font-size: 11px; margin-bottom: 14px; font-family: monospace; }\n #iaviz-389d0815 .tokens {\n   font-family: ui-monospace, "SF Mono", Menlo, Consolas, monospace;\n   font-size: 13px;\n   line-height: 1.85;\n   background: #fff;\n   border: 1px solid #ddd;\n   border-radius: 6px;\n   padding: 14px 16px;\n   white-space: pre-wrap;\n   word-break: break-word;\n }\n #iaviz-389d0815 .tok {\n   display: inline-block;\n   padding: 2px 4px;\n   margin: 1px 0;\n 